## Retrieval-Augmented Generation (RAG) AND Evaluation


### **Retrieval-Augmented Generation (RAG)**

Retrieval-Augmented Generation (RAG) is a powerful approach that combines information retrieval with text generation. Instead of relying solely on a language model’s internal knowledge, RAG retrieves relevant documents from an external source before generating responses. This enhances accuracy, reduces hallucinations, and provides up-to-date information.

In this lesson, we will implement RAG using **Together AI’s API** for both **document embeddings** and **text generation**. To follow along, get an API key from [Together AI](https://api.together.ai/).

---

### Setting Up the Environment

Before we begin, ensure you have the necessary libraries installed:

In [ ]:
%pip install pandas langchain chromadb langchain-together 

**Import the required libraries:**

- `pandas`: Used to load and manipulate the dataset.
- `uuid4`: Generates unique document IDs.
- `TogetherEmbeddings`: Embeds text into vector representations.
- `Chroma`: Stores and retrieves embedded documents efficiently.
- `ChatTogether`: Uses an AI model for text generation.
- `ChatPromptTemplate` and `StrOutputParser`: Create structured prompts and extract responses.
- `dotenv` :  This helps to load the environment variables like API Keys.

In [3]:
from langchain_chroma import Chroma
import pandas as pd
from uuid import uuid4
import os
import chromadb
import pathlib
from langchain_together import ChatTogether
from langchain_together import TogetherEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

**Loading the Dataset**


For this tutorial, we’ll use a CSV file containing data about MSMEs (Micro, Small, and Medium Enterprises) in Nigeria.

In [ ]:
msme = pd.read_csv("msme.csv")
msme 

When storing and retrieving documents in a Retrieval-Augmented Generation (RAG) system, we need more than just the text. Metadata and ID play a crucial role in managing, organizing, and retrieving relevant documents effectively.

In [5]:
documents = []
metadatas = []
ids = []

for index, row in msme.iterrows():    
    texts = str(row["Content"])
    title = row["Title"]
    sources = row["Sources"]
        
    content = texts
    metadata = {"Source":sources ,  "doc_title": title}
    id = f"{title}-{uuid4()}"

    documents.append(content)
    metadatas.append(metadata)
    ids.append(id)



In this tutorial, we are using the document title and source as metadata.

Also, we will be using the title and  UUID as as the ID.  
  
  - `Content`: The document’s main text.
  - `Title`: The document title.
  - `Sources`: source / links to each documents.

  - `documents`: Holds document texts.
  - `metadatas`: Metadata provides additional information about the document and also helps in filtering and refining search results.
  - `ids`: Ensures each document has a unique identifier, useful for tracking and retrieval.


In [26]:
#use this to comfirm the number documents to embed 
print(f"Total Documents to be embedded {len(documents)}\n")

Total Documents to be embedded 14



This loads the environment variables from a file named credentials.env, which contains the together API key.

In [6]:
load_dotenv()
together_api = os.environ["TOGETHER_API_KEY"]

**Setting Up ChromaDB for embeddings Storage**

chroma_dir: Gets the current working directory where the notebook is located.

chroma_path: Creates a folder named chroma_store to store the vector embeddings.

In [14]:
chroma_path = "chroma_store"


- ** For this tutorial We are using the togethercomputer/m2-bert-80M-32k-retrieval` embedding model** to convert documents into vector embeddings.
- ** you can check out More embedding models are  on Together AI (https://together.ai/)).

In [15]:
# Initialize the embedding model
embeddings = TogetherEmbeddings(
    model="togethercomputer/m2-bert-80M-32k-retrieval",
    api_key=together_api
)

- Creates a **vector database (Chroma)** for the documents.
- Adds **embedded texts**, **metadata**, and **IDs** to the database.

In [16]:
# Initialize ChromaDB
msmevdb = Chroma(
    collection_name="msme",
    embedding_function=embeddings,
    persist_directory=chroma_path
)


In [ ]:
# Add documents to the vector database
msmevdb.add_texts(
    texts=documents,
    metadatas=metadatas,
    ids=ids
)

In [17]:
#initialize the chatmodel
chatmodel = ChatTogether(
    api_key=together_api,
    model="meta-llama/Llama-3.3-70B-Instruct-Turbo",
    temperature=0
)

### The Retriever
Retrieving Documents for a Query

In [ ]:
# Maximal Marginal Relevance (MMR) for diverse and relevant results.
msme_retriever = msmevdb.as_retriever(search_type="mmr")

k defines the final number of documents that the retriever will return. use for concise and relevant results instead of retrieving too much data.

fetch_k determines how many documents are initially retrieved before filtering.

In [ ]:
msme_retriever = msmevdb.as_retriever(search_type="mmr", search_kwargs={'k': 4, 'fetch_k': 10})

Retrieves related documents based on `question`

In [ ]:
question = "How long does a business name reservation last in Nigeria?"
msme_retriever.invoke(question)

**Generating Responses Using Llama 3.3 70b  model**

- Let us use **Meta Llama 3.1 (8B)** for response generation.
- **`temperature= 0` ensures deterministic outputs.**
- You can try Other chat models are available on **Together AI (https://together.ai/))**.

In [ ]:
#initialize the chatmodel
chatmodel = ChatTogether(
    api_key=together_api,
    model="meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo",
    temperature=0
)

### **Prompting the LLM**

Creates a prompt template (prompt) with a system prompt and placeholders for the context and user question

In [ ]:
#using from template method
prompt = ChatPromptTemplate.from_template(
    """You are a business consultant providing insights on MSMEs in Nigeria.
You will be provided with the context: {context} to answer the user's question.
The context includes sections on understanding, starting, growing, and sustaining MSMEs, policies, and industry-specific information.
Provide a comprehensive response.
Include relevant sources or links from the context in your response at the end of each answer, include a statement: "To read more, check out this link: [insert link]."
Avoid unnecessary or unrelated details. Format the text output clearly and professionally in an HTML format.
question: {question}""")




Initializes the chat completion chat model (llm) to use for generating responses

Creates a simple output parser (parse_output) to parse the LLM output as a string

Chains all the above components using LangChain’s pipe ( | ) notation to create a simple RAG workflow (rag_chain)

In [ ]:
question = "How long does a business name reservation last in Nigeria?"

#retrieve the document
get_doc = msme_retriever.invoke(question)
# Prepare the input for the chain
input = {"context": get_doc, "question": question}

# Create the chain
chain = prompt | chatmodel | StrOutputParser()

answer = chain.invoke(input)
print(answer)

**Using from messages method**

- Separates **system instructions and user input** into structured messages.
- Can be adapted for other structured conversations.


In [ ]:
# Define the prompt using from_messages
prompt_messages = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a business consultant providing insights on MSMEs in Nigeria.
            You will be provided with the context: {context} to answer the user's question.
            The context includes sections on understanding, starting, growing, and sustaining MSMEs, policies, and industry-specific information.
            Provide a comprehensive response.
            Include relevant sources or links from the context in your response at the end of each answer, include a statement: "To read more, check out this link: [insert link]."
            Avoid unnecessary or unrelated details. Format the text output clearly and professionally in an HTML format."""
        ),
        ("human", "question: {question}"),
    ]
)

# Prepare the input for the chain
input = {"context": get_doc, "question": question}
# Create the chain
chain = prompt_messages | chatmodel | StrOutputParser()
# Generate and print the answer
result = chain.invoke(input)
print(result)

## **Other Retriever Techniques**

## Parent Document Retriever 


When preparing documents for LLMs:

Small chunks improve retrieval quality.
Large chunks maintain context for better generation.
Simple strategies like fixed or recursive splitting can't balance both. 

**Parent document retrieval solves this by:**
Embedding small chunks for retrieval.
Fetching larger chunks or source documents for context.

This ensures the LLM gets complete context, improving response quality. Useful for tasks needing expanded context.

#### import the necessary libraries 

We will require the following libraries for this method:

In [21]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document

We will be using the Parent Document Retriever library from LangChain.
  The easiest way to use your data with LangChain features is by converting them into LangChain document 

** This document consist of two attributes—page_content and metadata.**

In [42]:
#we will use the same MSME data set
msme = pd.read_csv("msme.csv")
msme_documents= []

for index, row in msme.iterrows():    
    # Extract content and metadata fields
    texts = str(row["Content"])
    title = str(row["Title"])
    sources = str(row["Sources"])
    
    # Create metadata dictionary
    metadata = {"source": sources, "doc_title": title}
    # Create Document object with content and metadata
    doc = Document(page_content=texts, metadata=metadata)
    
    msme_documents.append(doc)

print(f"Total Documents to be embedded: {len(msme_documents)}\n")

Total Documents to be embedded: 14



In [43]:
#This is to divides the documents into smaller chunks
split_msme = RecursiveCharacterTextSplitter(chunk_size=500)

Now, let's create a vector store for storing smaller chunks and an InMemoryStore to store the parent documents.

The InMemoryStore functions as a key-value pair data structure, where:

Each key is a unique UUID assigned to a parent document.

Each value is the actual text content of the corresponding parent document.

This structure ensures that while the program is running, the parent documents remain accessible in memory, allowing efficient retrieval and reconstruction of full documents when needed.


In [44]:
msme_vectorestore = Chroma(collection_name="msme_documents", embedding_function=embeddings)
# This store will retain parent documents for retrieval
store = InMemoryStore()

Initialize the Parent Document Retriever

In [45]:
retriever = ParentDocumentRetriever(
    vectorstore = msme_vectorestore , 
    docstore=store, 
    child_splitter=split_msme
)

Add Documents to Retriever

In [46]:
retriever.add_documents(msme_documents)

After adding, we can see there are 14 keys in the store. So 14 large documents have been added.

In [48]:
len(list(store.yield_keys()))

14

 This will only returns the small chunks

 Search for relevant document chunks using the vector store

In [ ]:
sub_docs = msme_vectorestore.similarity_search("What is the first step in setting up a business in Nigeria?")
sub_docs

: Retrieve Relevant Parent Documents

In [ ]:
retrieved_docs = retriever.get_relevant_documents("What is the first step in setting up a business in Nigeria?")

print(retrieved_docs[0].page_content)

### Generate Response 

we will use the same prompt above

In [51]:
question = "How long does a business name reservation last in Nigeria?"

#retrieve the document
retrieved_docs = retriever.get_relevant_documents(question)

# Prepare the input for the chain
input = {"context": retrieved_docs, "question": question}

# Create the chain
chain = prompt | chatmodel | StrOutputParser()

answer = chain.invoke(input)
print(answer)

<h2>Business Name Reservation in Nigeria</h2>

In Nigeria, a business name reservation is valid for 60 days from the date of application. This means that once a business name is reserved, the applicant has 60 days to register the business and obtain a Certificate of Incorporation.

If the applicant fails to register the business within the 60-day period, the reserved business name will be automatically cancelled, and it will be made available for reservation by another applicant.

It is essential to note that the 60-day period is a critical window for businesses to complete their registration process. Failure to do so may result in the loss of the reserved business name, which can have significant consequences for the business.

To avoid any issues, it is recommended that businesses apply for business name reservation and registration simultaneously to ensure a smooth and timely process.

To read more, check out this link: <a href="https://www.cac.gov.ng/">Corporate Affairs Commission 

### Multiple Query Retriever

This method enhances document retrieval by generating multiple variations of a user query, increasing the likelihood of retrieving relevant documents from a vector database

 We will prompt the LLM to create five different variations of the user’s query.
Helps overcome limitations of distance-based similarity search, ensuring that relevant documents aren't missed due to minor phrasing differences.

In [53]:

template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple type of the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. Do not add explanation or numbers to the question, only 
generate the questions.
Provide these alternative questions separated by newlines Original question: {question}"""
multiple_query_prompt = ChatPromptTemplate.from_template(template)

Alternative versions of the query, which are split into individual questions using split("\n") to create a list.

Since the LLM generates multiple queries as a block of text with line breaks, we split the text into individual lines to get distinct query variations.

Using .strip() prevents storing empty or whitespace-only strings in the final output.

In [ ]:
# question = "How long does a business name reservation last in Nigeria?"
question ="what is the repayment plan for loan from Development Bank of Nigeria"

multiple_queries = multiple_query_prompt | chatmodel | StrOutputParser() | (lambda x: [q for q in x.split("\n") if q.strip()])
multiple_answer = multiple_queries.invoke({"question" : question})
print(multiple_answer)

['what are the repayment plans for loans from Development Bank of Nigeria', 'what repayment plans does Development Bank of Nigeria offer for loans', 'Development Bank of Nigeria loan repayment plans', 'repayment plans for loans from Development Bank of Nigeria in Nigeria', 'Development Bank of Nigeria loan repayment terms']


Define the Retriever

In [57]:
msme_retriever = msmevdb.as_retriever()

Get Unique Documents


Let us Convert the documents into JSON strings using dumps() and also remove duplicates using (set), and convert them back to objects using loads().

This is to ensure the retrieved documents are unique.

In [58]:
from langchain.load import dumps, loads
def unique_doc (documents): 
    unique_doc = list(set(dumps(doc)for doc in documents))
    return [loads(doc) for doc in unique_doc]

Retrieving Documents Using Multiple Queries

With .map(), The msme_retriever processes each query separately, retrieving relevant documents for each one instead of handling all queries at once.

In [59]:
retrieval_chain = multiple_queries | msme_retriever.map() | unique_doc

Answer Generation Using Retrieved Documents

 We pass all the retrieved documents as context along with the original question to the llm

In [ ]:

prompt = ChatPromptTemplate.from_template(
    """You are a business consultant providing insights on MSMEs in Nigeria.
You will be provided with the context: {context} to answer the user's question.
The context includes sections on understanding, starting, growing, and sustaining MSMEs, policies, and industry-specific information.
Provide a detailed and comprehensive response.
Include relevant sources or links from the context in your response. At the end of each answer, include a statement: "To read more, check out this link: [insert link]."
Avoid unnecessary or unrelated details. Format the text output clearly and professionally in an HTML format in not more than 5 sentences.
question: {question}""")

multiple_query_input = ({"context" : retrieval_chain, "question": question})

chain = prompt| chatmodel| StrOutputParser()

answer = chain.invoke(multiple_query_input)
print(answer)  

<h2>Repayment Plan for Loan from Development Bank of Nigeria</h2>

<p>The Development Bank of Nigeria (DBN) offers a repayment plan for loans that is designed to be flexible and manageable for Micro, Small and Medium Enterprises (MSMEs). The repayment plan typically ranges from 3 to 7 years, depending on the loan amount and the borrower's cash flow projections.</p>

<p>According to the DBN's website, the repayment plan includes:</p>

<ul>
  <li>Equal monthly installments</li>
  <li>Interest rates that are competitive with other financial institutions</li>
  <li>Flexible repayment terms that can be adjusted to suit the borrower's needs</li>
</ul>

<p>It's worth noting that the repayment plan may vary depending on the specific loan product and the borrower's creditworthiness. MSMEs are advised to review the loan agreement carefully and seek clarification on any aspects of the repayment plan that are unclear.</p>

<p>To read more, check out this link: <a href="https://www.dbn.gov.ng/loans

### BM25 (BEST MATCHING 25)

**Problem:** Dense models sometimes overlook exact keywords in queries.  
**Solution:** Uses term frequency (TF-IDF) to rank documents with strong keyword overlap.

- Great for domains with specific terms (e.g., law, code)
- Useful for hybrid retrieval (dense + sparse)
- Fast and interpretable baseline

 Imports and Setup the neccessary libraries

In [19]:
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain.schema import Document
from nltk.tokenize import word_tokenize
import nltk

# Downloads tokenization rules
nltk.download("punkt_tab") 


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\YUSUF\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

**Load the Vector Store (Semantic Retriever)**

In [20]:
msmevdb = Chroma(
    collection_name="msme",
    embedding_function=embeddings,
    persist_directory=chroma_path
)

** Extract Raw Documents and Convert to LangChain Format**

You're fetching all stored documents + metadata from Chroma.

Then wrapping them as LangChain Document objects so they can be passed to other retrievers (like BM25).

In [21]:
question_docs = msmevdb.get(include=['documents', 'metadatas'])
question_documents = [Document(page_content=doc, metadata=meta) for doc, meta in zip(question_docs['documents'], question_docs['metadatas'])]

**Create Two Individual Retrievers**

In [22]:
question_semantic = msmevdb.as_retriever()
question_bm25 = BM25Retriever.from_documents(question_documents, preprocess_func=word_tokenize)

**Combine Both with EnsembleRetriever**

Combines both retrievers.

The weights=[0.5, 0.5] means equal influence from semantic and keyword-based scores.

You can adjust the weights to favor one over the other.

In [23]:
bm25_retriever = EnsembleRetriever(retrievers=[question_semantic, question_bm25], weights=[0.5, 0.5])

In [ ]:
question = "How do i register a construction business Nigeria"
results = bm25_retriever.get_relevant_documents(question )
results

In [25]:
# Prepare the input for the chain
input = {"context": results , "question": question}

# Create the chain
chain = prompt | chatmodel | StrOutputParser()

answer = chain.invoke(input)
print(answer)

To register a construction business in Nigeria, you need to follow these steps: register your business name with the Corporate Affairs Commission (CAC), obtain necessary licenses and permits from relevant authorities such as the Federal Ministry of Works and Housing, and register with professional associations like the Nigerian Institute of Builders. You will also need to obtain a Tax Identification Number (TIN) and register for Value Added Tax (VAT) with the Federal Inland Revenue Service (FIRS). <a href="https://cac.gov.ng/">To read more, check out this link: https://cac.gov.ng/</a>.


### RAG EVALUATION TECHNIQUES

#### RAG Evaluation using RAGAS

Retrieval-Augmented Generation (RAG) systems should not just “sound correct” — they must retrieve accurate, relevant, and faithful content.

We use RAGAS (Retrieval-Augmented Generation Assessment Suite) to measure how well the system performs on both retrieval and generation steps.

In [ ]:
from langchain_together import ChatTogether

**Sample Queries and References (Ground Truth)**

These represent typical user questions and what the system should ideally answer.

They form the evaluation dataset.

In [26]:
sample_queries = [
    "What are the classification criteria for MSMEs in Nigeria according to the National Policy on MSMEs?",
    "How do SMEDAN, BoI, and CBN define micro, small, and medium enterprises differently?",
    "What are the key challenges faced by MSMEs in Nigeria?",
    "Which sectors dominate micro enterprises in Nigeria, and what percentage do they represent?",
    "What government interventions or policies have been introduced to support MSMEs in Nigeria?"
]

expected_responses = [
    "The National Policy on MSMEs classifies enterprises as: Micro (<10 employees, assets <₦10M), Small (10-49 employees, assets ₦10M-₦100M), and Medium (50-199 employees, assets ₦100M-₦1B). Employment criteria take precedence if conflicts arise with asset thresholds.",
    "SMEDAN defines Micro (<10 employees, <₦5M assets), Small (10-49 employees, ₦5M-₦50M assets), and Medium (50-199 employees, ₦50M-₦500M assets). BoI and CBN use similar employee ranges but differ slightly in turnover/asset thresholds (e.g., BoI includes turnover <₦20M for Micro).",
    "Key challenges include poor access to credit, low market access, weak infrastructure, discriminatory legislation, and lack of technical skills. Micro enterprises also face funding constraints (reliance on personal savings/community funds).",
    "Micro enterprises are dominated by wholesale/retail trade (54.67%), followed by manufacturing (13.21%), agriculture (8.92%), and services (7.80%). The 2013 survey estimated 36.99 million micro enterprises in Nigeria.",
    "Interventions include SMEDAN (capacity building), BOI/CBN loan schemes, the Finance Act 2019/2020 (tax exemptions for small businesses), and programs like the National Economic Reconstruction Fund. Historical schemes date back to the 1970s (e.g., Agricultural Credit Guarantee Scheme)."
]

**Evaluate a Specific Retriever (e.g. BM25)**



In [27]:
#Let us evaluate one of our retriever. lets evaluate the BM25

relevant_docs=bm25_retriever.get_relevant_documents(question )

#### Run the Generation Chain

This chain creates the final LLM answer based on the retrieved context and the query.

In [28]:
# Prepare the input for the chain
input = {"context":  relevant_docs , "question": question}

# Create the chain
chain = prompt | chatmodel | StrOutputParser()


#### Build the Evaluation Dataset

For each query:

You retrieve documents

Run the LLM chain

Store the question, context, model answer, and reference answer in a list

In [47]:
dataset = []

for query, reference in zip(sample_queries, expected_responses):
    # Get relevant documents
    relevant_docs = bm25_retriever.get_relevant_documents(query)
    
    # Convert Document objects to strings for the context
    retrieved_contexts = [doc.page_content for doc in relevant_docs]
    
    # Get the model's response
    response = chain.invoke({"question": query, "context": retrieved_contexts})  # Adjust if your chain expects different input
    
    dataset.append({
        "user_input": query,        # Required by context_recall
        "contexts": retrieved_contexts,  # Used by context precision/relevancy
        "retrieved_contexts": retrieved_contexts,  # Required by context_recall
        "response": response,         # The generated answer
        "reference": reference     # Required by context_recall
    })


**Now, load the dataset into EvaluationDataset object.**

In [ ]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_list(dataset)

#### Set Up the LLM Evaluator


In [56]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

llm = ChatTogether(model="meta-llama/Llama-3.3-70B-Instruct-Turbo")
evaluator_llm = LangchainLLMWrapper(llm)




In [57]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness,  ResponseRelevancy

#### Choose RAGAS Metrics

LLMContextRecall: Measures if enough relevant info was retrieved

Faithfulness: Checks if the answer sticks to the retrieved info (no hallucinations)

FactualCorrectness: Verifies if the answer is factually accurate compared to the

##### Run Evaluation

In [58]:
result = evaluate(dataset=evaluation_dataset,metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],llm=evaluator_llm)
result

Evaluating: 100%|██████████| 15/15 [00:26<00:00,  1.78s/it]


{'context_recall': 0.9000, 'faithfulness': 0.9778, 'factual_correctness(mode=f1)': 0.5940}

**Result of our evaluation metrics**

Context Recall (0.90): The retriever is doing very well — it returns most of the relevant documents.

Faithfulness (0.98): The LLM sticks closely to the retrieved context — very low hallucination.

 Factual Correctness (0.59): Many answers miss details or don’t fully match the reference — accuracy needs improvement.